In [1]:
import pandas as pd
from pathlib import Path

In [2]:
department_df = pd.read_csv('../dataset/department.csv')
analyzer_df = pd.read_csv('../dataset/analyzer.csv')
order_item_df = pd.read_csv('../dataset/order-item.csv')
order_df = pd.read_csv('../dataset/order.csv')
patient_df = pd.read_csv('../dataset/patient.csv')
sample_state_df = pd.read_csv('../dataset/sample-state.csv')
sample_df = pd.read_csv('../dataset/sample.csv')
report_df = pd.read_csv('../dataset/report.csv', encoding='latin1')
test_df = pd.read_csv('../dataset/test.csv', encoding='latin1')
parameter_df = pd.read_csv('../dataset/parameter.csv', encoding='latin1')

In [3]:
order_df['created_at'] = pd.to_datetime(
    order_df['created_at'], unit='s', utc=True
)

sample_state_df['timestamp'] = pd.to_datetime(
    sample_state_df['timestamp'], unit='s', utc=True
)

In [4]:
patients = patient_df.rename(columns={
    'id': 'patient_id',
    'name': 'patient_name'
})

departments = department_df.rename(columns={
    'id': 'department_id',
    'name': 'department_name'
})

analyzers = analyzer_df.rename(columns={
    'id': 'analyzer_id',
    'department_id': 'analyzer_department_id',
    'name': 'analyzer_name'
})

tests = (
    test_df.rename(columns={
        'id': 'test_id',
        'name': 'test_name',
        'price': 'test_list_price',
        'analyser_id': 'analyzer_id'
    })
    .merge(departments, on='department_id', how='left', validate='many_to_one')
    .merge(analyzers, on='analyzer_id', how='left', validate='many_to_one')
)

orders = (
    order_df.rename(columns={
        'id': 'order_id',
        'created_at': 'order_created_at'
    })
    .merge(patients, on='patient_id', how='left', validate='many_to_one')
)

samples = sample_df.rename(columns={
    'id': 'sample_id',
    'order_id': 'sample_order_id',
    'patient_id': 'sample_patient_id',
    'type': 'sample_type'
})

sample_state = sample_state_df.rename(columns={'id': 'sample_state_id'})

In [5]:
item_analysis_df = (
    order_item_df.rename(columns={'id': 'order_item_id'})
    .merge(orders, on='order_id', how='left', validate='many_to_one')
    .merge(tests, on='test_id', how='left', validate='many_to_one')
    .merge(samples, on='sample_id', how='left', validate='many_to_one')
)

sample_state_analysis_df = (
    sample_state
    .merge(samples, on='sample_id', how='left', validate='many_to_one')
    .merge(
        orders,
        left_on='sample_order_id',
        right_on='order_id',
        how='left',
        validate='many_to_one'
    )
)

In [6]:
parameters = parameter_df.rename(columns={
    'id': 'parameter_id',
    'test_id': 'parameter_test_id',
    'name': 'parameter_name'
})

report_analysis_df = (
    report_df.rename(columns={'id': 'report_id'})
    .assign(dispatched_at=lambda df: pd.to_datetime(df['dispatched_at'], unit='s', utc=True))
    .merge(
        item_analysis_df,
        on='order_item_id',
        how='left',
        validate='many_to_one'
    )
    .merge(
        parameters,
        on='parameter_id',
        how='left',
        validate='many_to_one'
    )
)

In [7]:
output_dir = Path('../dataset/cleaned')
output_dir.mkdir(exist_ok=True)

item_analysis_df.to_csv(
    output_dir / 'item_analysis.csv',
    index=False,
    encoding='utf-8-sig'
)

sample_state_analysis_df.to_csv(
    output_dir / 'sample_state_analysis.csv',
    index=False,
    encoding='utf-8-sig'
)

report_analysis_df.to_csv(
    output_dir / 'report_analysis.csv',
    index=False,
    encoding='utf-8-sig'
)